In [1]:
# ── CONFIGURACION ──────────────────────────────────────────────────────────────
# Edita estas dos variables antes de ejecutar el notebook

ANOMES_CARPETA = "202604"   # Carpeta de entrada (ej. "202512"). None = la más reciente
ANOMES_OUTPUT  = None        # AñoMes del archivo de salida. None = igual que la carpeta
# ───────────────────────────────────────────────────────────────────────────────

In [2]:
import re
from pathlib import Path
import openpyxl
from openpyxl.utils import get_column_letter
from copy import copy
import pandas as pd
import xlrd
from openpyxl import Workbook
from datetime import datetime

BASE_DIR         = Path(r"C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad")
SHEET_CANDIDATES = ["2026", "KOTS-V"]


def get_folder(anomes_carpeta):
    if anomes_carpeta:
        folder = BASE_DIR / anomes_carpeta
        if not folder.exists():
            raise FileNotFoundError(f"No existe la carpeta: {folder}")
        return folder
    folders = sorted([f for f in BASE_DIR.iterdir() if f.is_dir() and re.fullmatch(r"\d{6}", f.name)])
    if not folders:
        raise FileNotFoundError(f"No se encontraron carpetas YYYYMM en {BASE_DIR}")
    return folders[-1]


def extract_tab_name(filepath):
    stem = filepath.stem
    match = re.search(r"\bAP\s+(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    match = re.search(r"(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return stem


def get_source_sheet(filepath, data_only=False):
    """Abre el archivo y devuelve la hoja 2025, KOTS-GT."""
    
    if filepath.suffix.lower() == ".xls":
        xls_book = xlrd.open_workbook(filepath)
        found_sheet = None
        
        for candidate in SHEET_CANDIDATES:
            for i in range(xls_book.nsheets):
                if xls_book.sheet_by_index(i).name == candidate:
                    found_sheet = xls_book.sheet_by_index(i)
                    break
            if found_sheet:
                break
        
        if not found_sheet:
            found_sheet = xls_book.sheet_by_index(0)
        
        new_wb = Workbook()
        new_ws = new_wb.active
        new_ws.title = found_sheet.name
        
        for row_idx in range(found_sheet.nrows):
            for col_idx in range(found_sheet.ncols):
                value = found_sheet.cell_value(row_idx, col_idx)
                new_ws.cell(row_idx + 1, col_idx + 1).value = value
        
        return new_ws
    else:
        wb = openpyxl.load_workbook(filepath, data_only=data_only)
        
        for candidate in SHEET_CANDIDATES:
            if candidate in wb.sheetnames:
                return wb[candidate]
        
        first = wb.sheetnames[0]
        print(f"  [aviso] Hoja {SHEET_CANDIDATES} no encontrada, usando '{first}'")
        return wb[first]


def find_header_row(ws):
    """Detecta encabezados buscando palabras clave tipicas."""
    keywords = ['kots', 'acc voucher', 'payment','due date','period','policy n°','profit center','share','cost center', 'invoice', 'account', 'ceding', 'date']
    
    for row_idx in range(1, min(50, ws.max_row + 1)):
        row_values = [str(ws.cell(row_idx, col).value).lower() 
                      for col in range(1, ws.max_column + 1) 
                      if ws.cell(row_idx, col).value is not None]
        
        keyword_count = sum(1 for kw in keywords if any(kw in val for val in row_values))
        
        if keyword_count >= 7:
            return row_idx
    
    for row_idx in range(1, min(50, ws.max_row + 1)):
        row_data = [ws.cell(row_idx, col).value for col in range(1, ws.max_column + 1)]
        non_empty = len([x for x in row_data if x is not None])
        if non_empty > 5:
            return row_idx
    
    return 1


def copy_cell_style(from_cell, to_cell):
    """Copia el estilo de una celda a otra."""
    if from_cell.has_style:
        to_cell.font = copy(from_cell.font)
        to_cell.border = copy(from_cell.border)
        to_cell.fill = copy(from_cell.fill)
        to_cell.number_format = copy(from_cell.number_format)
        to_cell.protection = copy(from_cell.protection)
        to_cell.alignment = copy(from_cell.alignment)


def copy_range_with_format(from_ws, to_ws, start_row, end_row, from_ws_values=None):
    """Copia un rango preservando formato. Valores de from_ws_values (data_only) si está disponible."""
    for row_idx in range(start_row, end_row + 1):
        for col_idx in range(1, from_ws.max_column + 1):
            from_cell = from_ws.cell(row_idx, col_idx)
            to_row = row_idx - start_row + 1
            to_cell = to_ws.cell(to_row, col_idx)
            
            # Usa valores calculados si está disponible, sino usa directamente
            if from_ws_values:
                value_cell = from_ws_values.cell(row_idx, col_idx)
                to_cell.value = value_cell.value
            else:
                to_cell.value = from_cell.value
            
            copy_cell_style(from_cell, to_cell)


def copy_column_widths(from_ws, to_ws):
    """Copia los anchos de columna."""
    for col_idx in range(1, from_ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        if from_ws.column_dimensions[col_letter].width:
            to_ws.column_dimensions[col_letter].width = from_ws.column_dimensions[col_letter].width

def clean_header_values(ws):
    """Limpia los encabezados de espacios raros, saltos de línea y caracteres especiales."""
    # Recolecta todas las posiciones de "Amount USD"
    amount_usd_positions = []
    
    for col_idx in range(1, ws.max_column + 1):
        header_cell = ws.cell(1, col_idx)
        if header_cell.value is not None:
            value = str(header_cell.value)
            # Reemplaza saltos de línea con espacios
            value = value.replace('\\n', ' ').replace('\\r', ' ')
            # Reemplaza múltiples espacios con uno solo
            value = ' '.join(value.split())
            # Quita espacios al inicio y final
            value = value.strip()
            
            # Guarda posición si es "Amount USD"
            if value == "Amount USD":
                amount_usd_positions.append(col_idx)
            
            header_cell.value = value
    
    # Renombra solo la ÚLTIMA "Amount USD" (la más a la derecha)
    if amount_usd_positions:
        last_col = amount_usd_positions[-1]
        ws.cell(1, last_col).value = "Amount USD (CORRECT)"

In [3]:
folder = get_folder(ANOMES_CARPETA)
anomes = ANOMES_OUTPUT if ANOMES_OUTPUT else folder.name

print(f"Carpeta de entrada : {folder}")
print(f"AñoMes del output  : {anomes}")

excel_files = [
    f for f in folder.iterdir()
    if f.suffix.lower() in (".xlsx", ".xls", ".xlsm")
    and not f.name.startswith("~$")
    and re.search(r"\bAP\s+\w+\s+vf", f.stem, re.IGNORECASE)
]

if not excel_files:
    raise FileNotFoundError(f"No se encontraron archivos 'AP * vf' en {folder}")

print(f"Archivos encontrados: {[f.name for f in excel_files]}\n")

output_path = folder / f"Payments_{anomes}.xlsx"
output_wb = openpyxl.Workbook()
output_wb.remove(output_wb.active)

for filepath in sorted(excel_files):
    tab_name = extract_tab_name(filepath)
    print(f"  Leyendo '{filepath.name}' -> pestaña '{tab_name}' ...", end=" ")
    
    # Lee dos veces: con formato y con valores calculados
    source_ws = get_source_sheet(filepath, data_only=False)  # Con formato
    source_ws_values = get_source_sheet(filepath, data_only=True)  # Con valores
    
    header_row = find_header_row(source_ws)
    end_row = source_ws.max_row
    
    num_rows = end_row - header_row + 1
    
    output_ws = output_wb.create_sheet(tab_name)
    
    copy_range_with_format(source_ws, output_ws, header_row, end_row, source_ws_values)
    copy_column_widths(source_ws, output_ws)
    clean_header_values(output_ws)  # Limpia los encabezados
    
    print(f"({num_rows} filas, desde fila {header_row})")

output_wb.save(output_path)
print(f"\nArchivo generado : {output_path}")

Carpeta de entrada : C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad\202604
AñoMes del output  : 202604
Archivos encontrados: ['260504 2026 04 AP AE vf.xlsx', '260504 2026 04 AP CL vf.xlsx', '260504 2026 04 AP VA vf.xlsx']

  Leyendo '260504 2026 04 AP AE vf.xlsx' -> pestaña 'AE' ... (59 filas, desde fila 21)
  Leyendo '260504 2026 04 AP CL vf.xlsx' -> pestaña 'CL' ... (276 filas, desde fila 23)
  Leyendo '260504 2026 04 AP VA vf.xlsx' -> pestaña 'VA' ... (929 filas, desde fila 24)

Archivo generado : C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad\202604\Payments_202604.xlsx
